# 04 — Document RAG

**Document RAG** is the workhorse strategy. When a query needs factual information from a curated knowledge base, the `DocumentRAGStrategy`:

1. Embeds the query and searches a **vector store** (e.g. Chroma, FAISS, Pinecone)
2. Retrieves the top‑k most similar document chunks
3. Builds a context from those chunks
4. Feeds the context + query to an LLM for answer generation

We'll explore `VectorRetriever` and `DocumentRAGStrategy`.

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.retrievers.vector_retriever import VectorRetriever
    from backend.adaptive_rag.strategies.document_rag_strategy import DocumentRAGStrategy
    print("✓ Imported VectorRetriever and DocumentRAGStrategy")
except ImportError as e:
    print(f"⚠ Backend import: {e}")
    print("→ Inline demo classes provided below")

## 2. Understanding VectorRetriever

`VectorRetriever` wraps any `vector_store` that supports `similarity_search_with_score`. It returns a list of dicts with `content`, `source`, and `relevance_score`.

Let's create a small in‑memory vector store to demonstrate:

In [ ]:
# Build a tiny in-memory vector store for demo
from langchain.embeddings import FakeEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document

fake_embeddings = FakeEmbeddings(size=768)

documents = [
    Document(page_content="Paris is the capital of France. It is known for the Eiffel Tower.",
             metadata={"source": "geography.txt"}),
    Document(page_content="The Eiffel Tower was built in 1889 for the World's Fair.",
             metadata={"source": "history.txt"}),
    Document(page_content="France is a country in Western Europe with a population of 68 million.",
             metadata={"source": "geography.txt"}),
    Document(page_content="Machine learning is a subset of artificial intelligence.",
             metadata={"source": "ai.txt"}),
    Document(page_content="Deep learning uses neural networks with multiple layers.",
             metadata={"source": "ai.txt"}),
    Document(page_content="Python is a high-level programming language created by Guido van Rossum.",
             metadata={"source": "programming.txt"}),
]

vector_store = FAISS.from_documents(documents, fake_embeddings)
retriever = VectorRetriever(vector_store)
print(f"✓ Vector store created with {len(documents)} documents")

## 3. Run a Retrieval

In [ ]:
import asyncio

async def demo_retrieval(query):
    docs = await retriever.retrieve(query, top_k=3)
    print(f"Query: {query}\n")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] Score: {d['relevance_score']:.4f}")
        print(f"      Source: {d['source']}")
        print(f"      Content: {d['content'][:120]}...")
        print()

asyncio.run(demo_retrieval("What is the capital of France?"))

## 4. Using DocumentRAGStrategy

`DocumentRAGStrategy` needs three things:
- A **retriever** (e.g. our `VectorRetriever`)
- An **LLM** (any object with `generate_with_context(query, context)`)
- A **grader** (for document relevance filtering)

Let's build a minimal LLM mock and see the full pipeline:

In [ ]:
class MockLLM:
    async def generate_with_context(self, query, context):
        # Simulate an LLM answer
        return f"Based on the documents, here is the answer to '{query[:50]}...'"

    async def generate(self, prompt):
        return f"Answer to: {prompt[:60]}..."

class MockGrader:
    async def grade(self, doc, query):
        return {"score": 0.9, "is_relevant": True, "explanation": "demo"}

llm = MockLLM()
grader = MockGrader()
strategy = DocumentRAGStrategy(retriever, llm, grader)
print(f"✓ Created DocumentRAGStrategy: {strategy.description()}")

## 5. Execute the Strategy

In [ ]:
async def run_strategy(query):
    result = await strategy.execute(query, top_k=3)
    print(f"Strategy  : {result['strategy']}")
    print(f"Query     : {result['query']}")
    print(f"Answer    : {result['answer'][:200]}")
    print(f"Sources   : {len(result['sources'])} documents")
    for s in result['sources']:
        print(f"           - {s['source']} (score: {s['score']:.2f})")

asyncio.run(run_strategy("Tell me about France and its capital"))

## 6. No Relevant Documents

The strategy gracefully handles the case where nothing is found:

In [ ]:
async def test_empty():
    result = await strategy.execute("Unicorns on Mars", top_k=3)
    print(f"Answer: {result['answer']}")
    print(f"Sources: {result['sources']}")

asyncio.run(test_empty())

## Key Points

| Component | Role |
|-----------|------|
| `VectorRetriever` | Wraps a vector store, returns ranked results |
| `DocumentRAGStrategy` | Orchestrates retrieve → filter → generate |
| Relevance filtering | Drops documents with `relevance_score ≤ 0.3` |

> **Next:** [05 — Web Search RAG](./05_Web_Search_RAG.ipynb) — live web retrieval